# SAP-DCG: End-to-End Training Pipeline on NVIDIA SSH Server (CTGAN & TVAE)

This notebook runs the full **4-Round Sequential Tuning Pipeline** (`SequentialTuningPipeline`) for both **CTGAN** and **TVAE** on a Linux NVIDIA CUDA server using the `sap-dcg-cuda` environment ([`env_cuda.yaml`](../env_cuda.yaml)), and exports all artifacts (`round2_best_weights.pt`, `tvae_per_class_models/`, `synthetic_samples_kept.csv`, `unsw_nb15_final_pipeline_train.csv`, `pipeline_tuning_summary.json`, `best_params.json`, `final_ids_model.joblib`, `evaluation_report.md`, and tuning plots) into `results/ctgan/result_<N>/` and `results/tvae/result_<N>/`.

### Headless Execution Tip (SSH)
To run this notebook unattended over SSH (survives disconnect):
```bash
conda activate sap-dcg-cuda
nohup jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 notebooks/train_ctgan_tvae_pipeline.ipynb > train_nb.log 2>&1 &
```

### Pipeline Stages Executed per Model
1. **Initialization:** Disaggregates Benign vs. Attack training flows and fits the 5-fold XGBoost OOF predictor, `signed_log1p` ENN ($k=3$), and local $k$-NN manifold-support gate ($k=5, \alpha=1.0$).
2. **Round 1:** Tunes SMOTE-NC multiplication factor $y_{\max} \in [5, 10, 15, 20, 25, 30, 40]$ with the generator in the loop.
3. **Round 2:** Trains the selected generator (`CTGAN` or `TVAE`) on the winning Round 1 configuration and checkpoints `round2_best_weights.pt`.
4. **Round 3:** Tunes the OOF confidence gate $\theta_{\text{OOF}} \in [0.4, 0.5, 0.6, 0.7, 0.8]$ with strict NetFlow invariant rejection, deduplication, ENN, and manifold gating; exports `synthetic_samples_kept.csv`.
5. **Final Round:** Optimizes the Two-Tier `FinalIDSModel` via Optuna, evaluates Validation/Test/Holdout partitions, and exports all final artifacts.

In [ ]:
import gc
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve SAP-DCG project root cleanly
cwd = Path(".").resolve()
candidate_roots = [
    cwd.parent if cwd.name in ("notebooks", "test") else cwd,
    cwd / "SAP-DCG",
    cwd.parent / "SAP-DCG",
]

ROOT = next((p.resolve() for p in candidate_roots if (p / "run_pipeline.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError(f"Could not locate SAP-DCG root containing run_pipeline.py in {[str(p) for p in candidate_roots]}")

for p in [ROOT, ROOT / "components", ROOT / "orchestration"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import torch
import ctgan
import sdv
import optuna
import xgboost as xgb
from orchestration.pipeline_orchestrator import SequentialTuningPipeline

print(f"Host:            {os.uname().nodename} ({os.uname().machine})")
print(f"Python:          {sys.version.split()[0]} ({sys.executable})")
print(f"SAP-DCG Root:    {ROOT}")
print(f"PyTorch:         {torch.__version__} | CUDA build: {torch.version.cuda} | GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Device:      {props.name} ({props.total_memory / (1024**3):.1f} GB VRAM)")
print(f"XGBoost:         {xgb.__version__} (USE_CUDA={xgb.build_info().get('USE_CUDA')})")
print(f"CTGAN / SDV:     {ctgan.__version__} / {sdv.__version__} | Optuna: {optuna.__version__}")

In [ ]:
# Pipeline hyperparameters (matching run_pipeline.py defaults)
T_MAX = 10000
EPSILON = 0.002
FIXED_GRADIENT_UPDATES = 42000
RANDOM_STATE = 42

# Set to None to use pipeline defaults:
#   CANDIDATE_YMAX = [5, 10, 15, 20, 25, 30, 40]
#   CANDIDATE_THRESHOLDS = [0.4, 0.5, 0.6, 0.7, 0.8]
CANDIDATE_YMAX = None
CANDIDATE_THRESHOLDS = None
COMP2_EPOCHS = None

RUN_CTGAN = True
RUN_TVAE = True

# Set to a specific folder name (e.g. "result_3") to reuse/overwrite, or None to auto-increment result_<N>
CTGAN_RUN_NAME = None
TVAE_RUN_NAME = None

FINAL_DIR = ROOT / "dataset"
RESULTS_ROOT = ROOT / "results"

def resolve_run_dir(results_root: Path, model_name: str, run_name: str = None) -> Path:
    model_dir = results_root / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    if run_name:
        out_dir = model_dir / run_name
        out_dir.mkdir(parents=True, exist_ok=True)
        return out_dir
    idx = 1
    while (model_dir / f"result_{idx}").exists():
        idx += 1
    out_dir = model_dir / f"result_{idx}"
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

ctgan_out_dir = resolve_run_dir(RESULTS_ROOT, "ctgan", CTGAN_RUN_NAME) if RUN_CTGAN else None
tvae_out_dir = resolve_run_dir(RESULTS_ROOT, "tvae", TVAE_RUN_NAME) if RUN_TVAE else None

print(f"Dataset Dir:     {FINAL_DIR}")
print(f"CTGAN Output:    {ctgan_out_dir}")
print(f"TVAE Output:     {tvae_out_dir}")

In [ ]:
train_path = FINAL_DIR / "dev_splits/NF-UNSW-NB15-v3_train.csv"
val_path = FINAL_DIR / "dev_splits/NF-UNSW-NB15-v3_val.csv"
test_path = FINAL_DIR / "dev_splits/NF-UNSW-NB15-v3_test.csv"
holdout_path = FINAL_DIR / "holdout_set/NF-UNSW-NB15-v3.csv"

for p in [train_path, val_path, test_path]:
    if not p.exists():
        raise FileNotFoundError(f"Required split not found: {p}")

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)
df_test = pd.read_csv(test_path)
df_holdout = pd.read_csv(holdout_path) if holdout_path.exists() else None

print(f"Loaded Train:   {len(df_train):,} rows (Attacks: {(df_train['Label'] == 1).sum():,})")
print(f"Loaded Val:     {len(df_val):,} rows")
print(f"Loaded Test:    {len(df_test):,} rows")
print(f"Loaded Holdout: {len(df_holdout):,} rows" if df_holdout is not None else "Holdout: None")

## 1. Train & Export CTGAN Pipeline (`results/ctgan/result_<N>/`)

In [ ]:
ctgan_results = None
if RUN_CTGAN:
    t0 = time.time()
    print("=" * 85)
    print(f"RUNNING CTGAN PIPELINE -> {ctgan_out_dir}")
    print("=" * 85)

    ctgan_orchestrator = SequentialTuningPipeline(
        t_max=T_MAX,
        epsilon=EPSILON,
        generator_type="ctgan",
        fixed_gradient_updates=FIXED_GRADIENT_UPDATES,
        random_state=RANDOM_STATE,
        final_dir=FINAL_DIR,
        output_dir=ctgan_out_dir,
    )
    ctgan_results = ctgan_orchestrator.run_pipeline(
        train_df=df_train,
        val_df=df_val,
        test_df=df_test,
        holdout_df=df_holdout,
        candidate_ymax=CANDIDATE_YMAX,
        candidate_thresholds=CANDIDATE_THRESHOLDS,
        comp2_epochs=COMP2_EPOCHS,
    )
    print(f"CTGAN pipeline finished in {(time.time() - t0) / 60:.2f} mins. Artifacts saved to: {ctgan_out_dir}")

    del ctgan_orchestrator
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 2. Train & Export TVAE Pipeline (`results/tvae/result_<N>/`)

In [ ]:
tvae_results = None
if RUN_TVAE:
    t0 = time.time()
    print("=" * 85)
    print(f"RUNNING TVAE PIPELINE -> {tvae_out_dir}")
    print("=" * 85)

    tvae_orchestrator = SequentialTuningPipeline(
        t_max=T_MAX,
        epsilon=EPSILON,
        generator_type="tvae",
        fixed_gradient_updates=FIXED_GRADIENT_UPDATES,
        random_state=RANDOM_STATE,
        final_dir=FINAL_DIR,
        output_dir=tvae_out_dir,
    )
    tvae_results = tvae_orchestrator.run_pipeline(
        train_df=df_train,
        val_df=df_val,
        test_df=df_test,
        holdout_df=df_holdout,
        candidate_ymax=CANDIDATE_YMAX,
        candidate_thresholds=CANDIDATE_THRESHOLDS,
        comp2_epochs=COMP2_EPOCHS,
    )
    print(f"TVAE pipeline finished in {(time.time() - t0) / 60:.2f} mins. Artifacts saved to: {tvae_out_dir}")

    del tvae_orchestrator
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 3. Verify Saved Artifacts & Compare CTGAN vs. TVAE Results

In [ ]:
runs = []
if RUN_CTGAN and ctgan_out_dir is not None:
    runs.append(("CTGAN", ctgan_out_dir))
if RUN_TVAE and tvae_out_dir is not None:
    runs.append(("TVAE", tvae_out_dir))

summary_rows = []
kept_counts = {}

for model_name, out_dir in runs:
    summary_path = out_dir / "pipeline_tuning_summary.json"
    kept_path = out_dir / "synthetic_samples_kept.csv"

    print(f"\n--- {model_name} Artifacts ({out_dir}) ---")
    for fname in [
        "pipeline_tuning_summary.json",
        "best_params.json",
        "round2_best_weights.pt",
        "synthetic_samples_kept.csv",
        "unsw_nb15_final_pipeline_train.csv",
        "final_ids_model.joblib",
        "evaluation_report.md",
    ]:
        fpath = out_dir / fname
        status = f"OK ({fpath.stat().st_size / (1024 * 1024):.2f} MB)" if fpath.exists() else "MISSING"
        print(f"  {fname:<36}: {status}")

    if summary_path.exists():
        with open(summary_path) as f:
            s = json.load(f)
        ho = s.get("holdout_performance") or {}
        r3 = s.get("tuning_rounds_summary", {}).get("round3", {})
        summary_rows.append({
            "Model": f"{model_name} ({out_dir.name})",
            "best_y_max": s["configuration"]["best_y_max"],
            "best_oof_thresh": r3.get("best_threshold"),
            "kept_syn": r3.get("kept_synthetic_samples"),
            "Val_Macro_F1": s["validation_performance"].get("macro_f1"),
            "Test_Macro_F1": s["test_performance"].get("macro_f1"),
            "Holdout_Macro_F1": ho.get("macro_f1"),
            "Holdout_T2_Rare_F1": ho.get("t2_rare_f1"),
            "Holdout_E2E_Rare_Recall": ho.get("e2e_rare_recall"),
            "Holdout_FAR": ho.get("false_alarm_rate"),
            "Holdout_MCC": ho.get("mcc"),
        })

    if kept_path.exists():
        df_k = pd.read_csv(kept_path)
        kept_counts[model_name] = df_k["Attack"].value_counts()

if summary_rows:
    display(pd.DataFrame(summary_rows).set_index("Model").round(4))
if kept_counts:
    display(pd.DataFrame(kept_counts).fillna(0).astype(int))